# Inference with Your Fine-Tuned Reasoning Model

This tutorial teaches you how to use your newly trained reasoning model for inference! We'll cover the complete inference pipeline from exporting your NeMo checkpoint to running streaming inference with your fine-tuned LoRA adapter.

Building on the previous tutorial where you fine-tuned a [Meta LLaMA 3.1–8B Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) model with LoRA adapters, you'll now learn how to:

### ✅ What You'll Learn
1. **Export NeMo checkpoints** to HuggingFace format (both full models and LoRA adapters)
2. **Load LoRA adapters** with the base model using PEFT (Parameter Efficient Fine-Tuning)
3. **Run inference** with your reasoning model, including streaming generation
4. **Compare responses** between base model and fine-tuned model

### 🧰 Tools and Resources
* [PEFT Library](https://github.com/huggingface/peft) for loading LoRA adapters
* [Transformers](https://github.com/huggingface/transformers) for model inference
* [NeMo Framework Export API](https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/features/export.html)

## 📌 Prerequisites

* A trained NeMo checkpoint from the previous tutorial
* Access to the base LLaMA 3.1-8B Instruct model
* Basic understanding of LoRA/PEFT concepts

Let's dive in! 🚀


## 📋 Step 1. Export NeMo Checkpoint to HuggingFace Format

Before we can use our trained model for inference, we need to export it from NeMo's internal format to HuggingFace format. NeMo 2.0 provides a convenient export API that handles this conversion.

### 🎯 Understanding Export Formats

NeMo supports different export targets depending on your training approach:

| Export Target | Use Case | Output |
|---------------|----------|--------|
| `"hf"` | Full model fine-tuning | Complete HuggingFace model |
| `"hf-peft"` | LoRA/PEFT training | Base model + LoRA adapter files |

Since we trained with **LoRA adapters** in the previous tutorial, we'll use `target="hf-peft"` to export our adapter weights.

### 🔧 Export Process

The export process involves:
1. Finding your latest checkpoint from the training run
2. Using NeMo's `export_ckpt()` API to convert to HuggingFace format
3. Saving the LoRA adapter files that can be loaded with PEFT

Here's the key export code from our utility script:

```python
from nemo.collections.llm.api import export_ckpt

# Export LoRA adapter to HuggingFace PEFT format
result = export_ckpt(
    path=checkpoint_path,           # Path to your NeMo checkpoint
    target="hf-peft",              # Export LoRA adapter format
    output_path=output_path,        # Where to save HF format
    overwrite=True                  # Overwrite existing files
)
```

⚠️ **Important**: For LoRA training, the export creates adapter files (like `adapter_config.json`, `adapter_model.safetensors`) that work with the original base model, not a standalone model file.

Let's set up our paths and run the export process:


In [ ]:
import os
from pathlib import Path

# Configuration paths
CHECKPOINT_SEARCH_DIR = "/workspace/output"  # Directory containing your training results
BASE_MODEL_PATH = "/workspace/models/Llama-3.1-8B-Instruct"  # Base LLaMA model
EXPORTED_ADAPTER_PATH = "/workspace/models/llama-3-8b-lora-hf"  # Where to save exported adapter

# Verify paths exist
assert os.path.exists(CHECKPOINT_SEARCH_DIR), f"Checkpoint directory not found: {CHECKPOINT_SEARCH_DIR}"
assert os.path.exists(BASE_MODEL_PATH), f"Base model not found: {BASE_MODEL_PATH}"

print(f"✅ Checkpoint search directory: {CHECKPOINT_SEARCH_DIR}")
print(f"✅ Base model path: {BASE_MODEL_PATH}")
print(f"✅ Export target: {EXPORTED_ADAPTER_PATH}")


### 🔄 Run the Export Process

Now let's export our checkpoint. We'll use our utility script which handles finding the latest checkpoint and performing the export:   


In [ ]:
# Code Sample - pretrained model already exported to huggingface safetensors format for our nim workshop!

# Import our utility functions

# import sys
# sys.path.append('./scripts')
# from reasoning_inference_utils import export_checkpoint_to_hf
# 
# # Export checkpoint to HuggingFace format
# print("🔄 Starting checkpoint export...")
# success = export_checkpoint_to_hf(
#     checkpoint_search_dir=CHECKPOINT_SEARCH_DIR,
#     output_path=EXPORTED_ADAPTER_PATH
# )
# 
# if success:
#     print(f"\n✅ Successfully exported LoRA adapter to: {EXPORTED_ADAPTER_PATH}")
#     print("📁 Export contents:")
#     if os.path.exists(EXPORTED_ADAPTER_PATH):
#         for item in os.listdir(EXPORTED_ADAPTER_PATH):
#             print(f"   📄 {item}")
# else:
#     print("❌ Export failed. Please check your checkpoint path.")


## 🔧 Step 2. Load Your Fine-Tuned Model with PEFT

Now that we have our LoRA adapter in HuggingFace format, let's load it for inference. With LoRA adapters, we follow a two-step process:

1. **Load the base model** (LLaMA 3.1-8B Instruct)
2. **Apply the LoRA adapter** using PEFT

This approach is memory-efficient because we only load the small adapter weights (typically <100MB) on top of the base model.

### 🎯 Understanding the Loading Process

```python
# Step 1: Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,  # Use half precision for efficiency
    device_map="auto"          # Automatically distribute across GPUs
)

# Step 2: Apply LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    is_trainable=False         # We're doing inference, not training
)
```

Let's implement this:


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig
import warnings
warnings.filterwarnings("ignore")

print("🔧 Loading base model and LoRA adapter...")

# Step 1: Load the base LLaMA 3.1-8B Instruct model
print(f"📥 Loading base model from: {BASE_MODEL_PATH}")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.float16,  # Use half precision for memory efficiency
    device_map="auto"          # Automatically distribute across available GPUs
)

# Load tokenizer from base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)

# Set pad token if not already set (required for batched inference)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Base model and tokenizer loaded successfully!")

# Step 2: Load and apply the LoRA adapter
print(f"📥 Loading LoRA adapter from: {EXPORTED_ADAPTER_PATH}")

# First, load the PEFT configuration to verify it's correct
peft_config = PeftConfig.from_pretrained(EXPORTED_ADAPTER_PATH)
print(f"✅ PEFT config loaded: {peft_config.peft_type} (rank={peft_config.r}, alpha={peft_config.lora_alpha})")

# Apply the LoRA adapter to the base model
fine_tuned_model = PeftModel.from_pretrained(
    base_model, 
    EXPORTED_ADAPTER_PATH, 
    is_trainable=False  # Set to False for inference mode
)

print("✅ LoRA adapter loaded and applied successfully!")
print(f"🎯 Model ready for reasoning inference!")


In [ ]:
# Lets see how the model is allocated to GPUs
from reasoning_inference_utils import get_gpu_inference_allocation
get_gpu_inference_allocation(base_model)


### 🔍 Model Architecture Overview

Let's examine what we've loaded:


In [ ]:
# Display model information
print("📊 Model Information:")
print(f"   Base model type: {type(base_model).__name__}")
print(f"   Fine-tuned model type: {type(fine_tuned_model).__name__}")
print(f"   Device: {next(fine_tuned_model.parameters()).device}")
print(f"   Data type: {next(fine_tuned_model.parameters()).dtype}")

# Check LoRA adapter details
print(f"\n🎛️ LoRA Adapter Details:")
print(f"   Adapter type: {peft_config.peft_type}")
print(f"   LoRA rank (r): {peft_config.r}")
print(f"   LoRA alpha: {peft_config.lora_alpha}")
print(f"   Target modules: {peft_config.target_modules}")
#print(f"   Trainable parameters: {fine_tuned_model.get_nb_trainable_parameters():,}")


## 🚀 Step 3. Run Inference with Your Reasoning Model

Now for the exciting part - let's use our fine-tuned reasoning model! We'll implement both regular generation and experiment with streaming inference.

### 🎯 Understanding Prompt Formatting

Our reasoning model was trained with specific system prompts that control reasoning behavior:
- **"detailed thinking on"** - Enables step-by-step reasoning
- **"You are a helpful assistant."** - Standard assistant behavior

We'll use the chat template format that LLaMA 3.1 expects:

```python
messages = [
    {"role": "system", "content": "detailed thinking on"},
    {"role": "user", "content": "Your question here"}
]
```

Let's start with our utility functions for inference:


In [ ]:
# Import our inference utilities
from reasoning_inference_utils import (
    format_chat_prompt,
    generate_response,
    generate_streaming_response
)

# Test a simple reasoning prompt
test_prompt = "How many r's are in the word 'strawberry'?"

print(f"🧪 Testing inference with prompt: {test_prompt}")
print("\n" + "="*80)


### 🤖 Standard Inference

Let's start with standard (non-streaming) inference:


In [ ]:
# Generate response with reasoning enabled
print("🧠 Fine-tuned model with reasoning:")
print("-" * 40)

response = generate_response(
    model=fine_tuned_model,
    tokenizer=tokenizer,
    prompt=test_prompt,
    system_prompt="detailed thinking on",
    max_new_tokens=512,
    temperature=0.1
)

print(f"Response: {response}")
print("\n" + "="*80)


### 🌊 Streaming Inference

Now let's experiment with streaming inference, where we get tokens as they're generated:


In [ ]:
print("🌊 Streaming inference:")
print("-" * 40)
print(f"Question: {test_prompt}")
print("Answer: ", end="", flush=True)

# Stream the response token by token
for token in generate_streaming_response(
    model=fine_tuned_model,
    tokenizer=tokenizer,
    prompt=test_prompt,
    system_prompt="detailed thinking on",
    max_new_tokens=512,
    temperature=0.1
):
    print(token, end="", flush=True)

print("\n\n" + "="*80)


### 🔍 Compare Base vs Fine-Tuned Model

Let's compare how the base model and fine-tuned model handle the same reasoning task:


In [ ]:
# Import comparison utility
from reasoning_inference_utils import compare_models

# Compare both models on a reasoning task
comparison_prompt = "Which number is bigger? 10.119 or 10.19?"

print(f"🔍 Comparing models on: {comparison_prompt}")
print("\n" + "="*100)

base_response, ft_response = compare_models(
    base_model=base_model,
    fine_tuned_model=fine_tuned_model,
    tokenizer=tokenizer,
    prompt=comparison_prompt,
    max_new_tokens=512
)

print("\n" + "="*100)


### 🧩 Try More Complex Reasoning

Let's test our model on a more complex probability problem:


In [ ]:
complex_prompt = """
You have three boxes: 
- Box A contains two white balls
- Box B contains two black balls  
- Box C contains one white and one black ball

You pick a box at random and draw one ball at random—it's white. 
Without putting it back, you draw the second ball from the same box. 
What is the probability the second ball is white? 
Show your reasoning step by step.
""".strip()

print("🧩 Complex reasoning test:")
print("-" * 40)
print(f"Problem: {complex_prompt}")
print("\n🤖 Fine-tuned model response:")
print("-" * 40)

complex_response = generate_response(
    model=fine_tuned_model,
    tokenizer=tokenizer,
    prompt=complex_prompt,
    system_prompt="detailed thinking on",
    max_new_tokens=1024,
    temperature=0.1
)

print(complex_response)


### 🎛️ Interactive Inference

Now let's create an interactive cell where you can test your own prompts:


In [ ]:
# Interactive inference cell
print("🎯 Interactive Inference")
print("Enter your own prompts to test the reasoning model!")
print("(Leave blank and press Enter to skip)")
print("\n" + "-"*50)

while True:
    try:
        print("Type cntl-c + return to text box to exit")
        user_prompt = input("\n🤔 Your question: ").strip()
        if not user_prompt:
            print("👋 Exiting interactive mode.")
            break
            
        print("\n🤖 Model response:")
        print("-" * 30)
        
        response = generate_response(
            model=fine_tuned_model,
            tokenizer=tokenizer,
            prompt=user_prompt,
            system_prompt="detailed thinking on",
            max_new_tokens=512,
            temperature=0.1
        )
        
        print(response)
        print("\n" + "-"*50)
        
    except KeyboardInterrupt:
        print("\n👋 Exiting interactive mode.")
        break
    except Exception as e:
        print(f"❌ Error: {e}")


## 📊 Performance Analysis

Let's analyze some key metrics about our inference setup:


In [ ]:
import time

# Benchmark inference speed
print("⚡ Performance Benchmark")
print("-" * 40)

benchmark_prompt = "Explain why the sky is blue in simple terms."

# Time the inference
start_time = time.time()
benchmark_response = generate_response(
    model=fine_tuned_model,
    tokenizer=tokenizer,
    prompt=benchmark_prompt,
    system_prompt="detailed thinking on",
    max_new_tokens=256,
    temperature=0.1
)
end_time = time.time()

# Calculate metrics
generation_time = end_time - start_time
response_length = len(benchmark_response.split())
tokens_per_second = response_length / generation_time

print(f"📝 Response length: {response_length} words")
print(f"⏱️ Generation time: {generation_time:.2f} seconds")
print(f"🚀 Speed: {tokens_per_second:.1f} words/second")
print(f"\n📄 Response: {benchmark_response[:200]}...")


## 🎉 Congratulations! You've Mastered NeMo Inference!

You've successfully learned how to:

✅ **Export NeMo checkpoints** to HuggingFace format using `target="hf-peft"` for LoRA adapters

✅ **Load LoRA adapters** efficiently with PEFT, combining base models with fine-tuned adapters

✅ **Run inference** with your reasoning model, including both standard and streaming generation

✅ **Compare model behavior** between base and fine-tuned versions

✅ **Optimize inference** with proper prompt formatting and system messages

### 🚀 Next Steps

* 🔧 **Optimize for Production**: Use with Universal NIM to run production grade inference!
* 📊 **Evaluate Performance**: Test your model on reasoning benchmarks like GSM8K or MMLU
* 🔄 **Iterate and Improve**: Use inference results to refine your training data and approach.  Build your own eval datasets!

### 💡 Key Takeaways

1. **LoRA adapters** provide an efficient way to fine-tune large models with minimal resource requirements
2. **Proper prompt formatting** is crucial for getting the best performance from instruction-tuned models
3. **System prompts** can control model behavior (like enabling/disabling reasoning)
4. **Streaming inference** provides better user experience for interactive applications

Great job completing the NeMo reasoning model pipeline! 🎯
